# Google News RSS 수집 — scored=0 8개 기업 복구

**목적**: GDELT BigQuery에는 title 필드가 없어서 scored=0 상태인 8개 기업
(Honda/Ford/Glencore/Albemarle/Lucid/BASF/Asahi Kasei/SQM)의
title-bearing 뉴스 데이터를 Google News RSS로 복구.

**실행 환경**: Colab (news.google.com이 Cowork VM에서 차단됨)

**산출물**: `data/raw/supplement/google_news_zero8.parquet`

**소요 시간**: 반기 × 16개 × locale × 3~4 queries × 8 companies ≈ 1000~2000 requests → 약 15~30분

## 0. Drive mount (선택)

In [ ]:
# Colab + Drive 사용 시
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/나비효과'  # 실제 경로에 맞게 수정
    import os
    os.chdir(PROJECT_DIR)
    print('cwd:', os.getcwd())
except ImportError:
    import os
    print('Local mode, cwd:', os.getcwd())

## 1. 의존성

In [ ]:
!pip install -q pandas requests pyarrow

## 2. 수집 로직 (scripts/collect_google_news_zero8.py 와 동일)

In [ ]:
import hashlib, logging, time
from datetime import datetime
from pathlib import Path
from urllib.parse import quote_plus, urlparse
from xml.etree import ElementTree
import pandas as pd, requests

log = logging.getLogger('gn_zero8')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

TARGETS = {
    '7267.T': {
        'name': 'Honda Motor',
        'queries': {
            'en': ['"Honda Motor" battery EV', '"Honda" electric vehicle', '"Honda" EV supply chain'],
            'jp': ['ホンダ 電池', 'ホンダ EV', '本田技研 電動化'],
        },
        'locales': [('en', 'US', 'US:en'), ('ja', 'JP', 'JP:ja')],
    },
    'F': {
        'name': 'Ford Motor',
        'queries': {
            'en': ['"Ford Motor" battery', '"Ford" EV F-150 Lightning', '"Ford" battery plant', '"Ford" Mustang Mach-E'],
        },
        'locales': [('en', 'US', 'US:en')],
    },
    'GLEN.L': {
        'name': 'Glencore',
        'queries': {
            'en': ['"Glencore" cobalt', '"Glencore" lithium', '"Glencore" battery material', '"Glencore" nickel mining'],
        },
        'locales': [('en', 'US', 'US:en'), ('en', 'GB', 'GB:en')],
    },
    'ALB': {
        'name': 'Albemarle',
        'queries': {
            'en': ['"Albemarle" lithium', '"Albemarle Corp" battery', '"Albemarle" hydroxide', '"Albemarle" Chile Australia'],
        },
        'locales': [('en', 'US', 'US:en')],
    },
    'LCID': {
        'name': 'Lucid Motors',
        'queries': {
            'en': ['"Lucid Motors" battery', '"Lucid Group" EV', '"Lucid Air"', '"Lucid Motors" supply chain'],
        },
        'locales': [('en', 'US', 'US:en')],
    },
    'BAS.DE': {
        'name': 'BASF',
        'queries': {
            'en': ['"BASF" battery material', '"BASF" cathode', '"BASF" lithium', '"BASF SE" EV'],
            'de': ['BASF Batterie', 'BASF Kathode', 'BASF Elektromobilität'],
        },
        'locales': [('en', 'US', 'US:en'), ('de', 'DE', 'DE:de')],
    },
    '3407.T': {
        'name': 'Asahi Kasei',
        'queries': {
            'en': ['"Asahi Kasei" separator battery', '"Asahi Kasei" lithium', '"Asahi Kasei" Hipore'],
            'jp': ['旭化成 セパレータ', '旭化成 電池', '旭化成 ハイポア'],
        },
        'locales': [('en', 'US', 'US:en'), ('ja', 'JP', 'JP:ja')],
    },
    'SQM': {
        'name': 'SQM Lithium',
        'queries': {
            'en': ['"SQM" lithium Chile', '"Sociedad Quimica" lithium', '"SQM" brine Atacama', '"SQM" battery'],
            'es': ['SQM litio Chile', 'Sociedad Quimica Minera litio', 'SQM Atacama'],
        },
        'locales': [('en', 'US', 'US:en'), ('es', 'CL', 'CL:es-419')],
    },
}

HALVES = []
for year in range(2018, 2026):
    HALVES.append((f'{year}-01-01', f'{year}-06-30'))
    HALVES.append((f'{year}-07-01', f'{year}-12-31'))

print(f'Targets: {len(TARGETS)}, Halves: {len(HALVES)}')

In [ ]:
def fetch_rss(query, after, before, hl, gl, ceid, max_retry=3):
    q = f'{query} after:{after} before:{before}'
    url = f'https://news.google.com/rss/search?q={quote_plus(q)}&hl={hl}&gl={gl}&ceid={ceid}'
    for attempt in range(max_retry):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code != 200:
                if r.status_code in (429, 503):
                    time.sleep(5 + attempt * 5)
                    continue
                return []
            root = ElementTree.fromstring(r.content)
            items = []
            for item in root.findall('.//item'):
                t = item.find('title'); l = item.find('link'); p = item.find('pubDate')
                title = t.text.strip() if t is not None and t.text else ''
                link = l.text.strip() if l is not None and l.text else ''
                pub = p.text.strip() if p is not None and p.text else ''
                if not link or not title: continue
                event_time = None
                for fmt in ('%a, %d %b %Y %H:%M:%S %Z', '%a, %d %b %Y %H:%M:%S %z'):
                    try: event_time = datetime.strptime(pub, fmt); break
                    except: continue
                items.append({
                    'title': title, 'url': link, 'event_time': event_time,
                    'source_domain': urlparse(link).netloc.replace('www.', ''),
                })
            return items
        except Exception as e:
            if attempt == max_retry - 1:
                print(f'fetch_rss failed: {e}')
            else:
                time.sleep(2 + attempt * 2)
    return []


def collect_one(cid, info):
    lang_map = {'en':'en','ja':'jp','de':'de','es':'es'}
    all_items = []
    for hl, gl, ceid in info['locales']:
        lang_key = lang_map.get(hl[:2], hl[:2])
        if lang_key not in info['queries']: continue
        queries = info['queries'][lang_key]
        print(f'    locale={hl}-{gl} ({lang_key}): {len(queries)} queries × {len(HALVES)} halves')
        for q in queries:
            for after, before in HALVES:
                items = fetch_rss(q, after, before, hl, gl, ceid)
                for it in items:
                    it['company_id'] = cid
                    it['lang'] = lang_key
                all_items.extend(items)
                time.sleep(0.8)
    return all_items

## 3. 실행 (약 15~30분)

In [ ]:
all_rows = []
for i, (cid, info) in enumerate(TARGETS.items(), 1):
    print(f'[{i}/{len(TARGETS)}] {cid} ({info["name"]})')
    items = collect_one(cid, info)
    before_n = len(items)
    seen = set(); uniq = []
    for it in items:
        if it['url'] in seen: continue
        seen.add(it['url']); uniq.append(it)
    print(f'  → {before_n:,} raw, {len(uniq):,} unique')
    all_rows.extend(uniq)

print(f'\nTotal collected: {len(all_rows):,}')

## 4. DataFrame 구성 + 저장

In [ ]:
rows = []
for it in all_rows:
    rows.append({
        'event_id': hashlib.md5(f"{it['url']}|{it.get('company_id','')}".encode()).hexdigest()[:16],
        'event_time': it['event_time'],
        'url': it['url'],
        'title': it['title'],
        'risk_types': 'other',
        'severity': 2.0,
        'alias': TARGETS[it['company_id']]['name'],
        'company_id': it['company_id'],
        'country_ids': '',
        'tone': 0.0,
        'source_domain': it['source_domain'],
        'collection_type': 'google_news_zero8',
        'lang': it.get('lang', ''),
    })
df = pd.DataFrame(rows)
n_before = len(df)
df = df.drop_duplicates(subset=['url','company_id'], keep='first').reset_index(drop=True)
df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce')
try: df['event_time'] = df['event_time'].dt.tz_localize(None)
except: pass
print(f'Total: {n_before:,} → dedup: {len(df):,}')

print('\n=== Per-Company Recovery ===')
for cid in TARGETS.keys():
    n = (df['company_id'] == cid).sum()
    print(f'  {cid:<10} {TARGETS[cid]["name"]:<15}: {n:>5,}')

print('\n=== Year distribution ===')
if len(df) > 0:
    df['year'] = df['event_time'].dt.year
    print(df.groupby('year').size())

In [ ]:
out = Path('data/raw/supplement/google_news_zero8.parquet')
out.parent.mkdir(parents=True, exist_ok=True)
df.drop(columns=['year'], errors='ignore').to_parquet(out, index=False)
print(f'✅ Saved: {out} ({len(df):,} rows)')

## 5. 샘플 확인

In [ ]:
# 기업별 상위 3개 샘플
for cid in TARGETS.keys():
    sub = df[df['company_id']==cid].head(3)
    if len(sub) == 0:
        print(f'\n[{cid}] (none)')
        continue
    print(f'\n[{cid}] {TARGETS[cid]["name"]}')
    for _, r in sub.iterrows():
        print(f'  {r["event_time"]} [{r["lang"]}] {r["title"][:80]}')